# 04 · `[컴바인 + 경기 통합]` 결측·스케일링 방식 비교 & 누수 방지 파이프라인

이 노트북은 **컴바인 피처(T2) + 경기 타깃(T3·T4)을 통합**한 뒤, 모든 전처리 선택을 감이 아닌 **Train(408명) 5×5 층화 CV 숫자로 직접 비교·검증**한다.
- **절대 원칙**: 테스트셋(`test_ids.csv`, 102명)은 2절에서 격리한 뒤 **이 노트북 끝까지 절대 열지 않는다**.
- **타깃 누수 방지**: 시즌 코호트 상대 타깃(`*_cohort_z`)은 CV fold마다 **학습 fold로만** 기준을 잡는다 (`RelativeTargetEncoder`).
- **비교 판정**: 모든 비교는 같은 25개 fold 점수를 짝지어 기준 설정 대비 `delta_mean`, `wins`, 보정 표준오차(`delta_se_corrected`)로 판정한다. `delta_mean > 2 × SE`일 때만 "우위".
- **비교 모델**: 이미 학습하신 `Ridge / Lasso / ElasticNet / PCA`만 사용해 전처리 효과를 투명하게 확인한다.

### [0] 실행 환경 부트스트랩 (로컬 / Colab 겸용)
- **데이터 범위**: `공통 환경`
- **작업 단계**: `환경 설정`
- **목적 및 원리**: `StopIteration` 없이 프로젝트 루트를 탐색하고 모듈을 불러온다.
- **확인할 내용**: `9/9 CSVs ready`.

In [ ]:
# =====================================================================
# [데이터: 공통] [단계: 환경 설정] 로컬/VS Code Colab 커널 겸용 루트 탐색
# =====================================================================
import importlib.util
import os
import sys
from pathlib import Path

# Google Drive 내 다른 경로에 저장소/데이터가 있다면 아래 주석을 풀고 경로를 지정하세요.
# os.environ["NFL_PROJECT_ROOT"] = "/content/drive/MyDrive/nfl-bdb2027"
# os.environ["NFL_DATA_DIR"] = "/content/drive/MyDrive/nfl-bdb2027-data"

def _is_colab_kernel() -> bool:
    if (
        "google.colab" in sys.modules
        or "COLAB_RELEASE_TAG" in os.environ
        or "COLAB_GPU" in os.environ
    ):
        return True
    try:
        return importlib.util.find_spec("google.colab") is not None
    except (ModuleNotFoundError, ValueError):
        return False

def find_project_root() -> Path:
    candidates: list[Path] = []
    if env_root := os.environ.get("NFL_PROJECT_ROOT"):
        candidates.append(Path(env_root).expanduser())

    cwd = Path.cwd()
    candidates.extend([cwd, *cwd.parents, cwd / "nfl-bdb2027"])

    in_colab = _is_colab_kernel()
    drive_root = Path("/content/drive")
    mydrive = drive_root / "MyDrive"

    if in_colab:
        if not mydrive.exists():
            try:
                from google.colab import drive
                drive.mount(str(drive_root), force_remount=False)
            except Exception as exc:
                print(f"[warn] Google Drive 자동 마운트 실패: {exc}")

        candidates.extend([
            Path("/content/nfl-bdb2027"),
            mydrive / "nfl-bdb2027",
            mydrive / "Colab Notebooks" / "nfl-bdb2027",
        ])

    seen: set[Path] = set()
    for p in candidates:
        if p in seen:
            continue
        seen.add(p)
        if (p / "src" / "config.py").exists():
            return p.resolve()

    if in_colab:
        skip = {"drive", "sample_data", ".config", ".git", ".venv", "__pycache__"}
        depth1_dirs: list[Path] = []
        for base in (Path("/content"), mydrive, drive_root / "Shareddrives"):
            if not base.exists():
                continue
            try:
                children = [
                    d for d in base.iterdir()
                    if d.is_dir() and d.name not in skip and not d.name.startswith(".")
                ]
            except OSError:
                continue
            depth1_dirs.extend(children)
            for d in children:
                for cand in (d, d / "nfl-bdb2027"):
                    if (cand / "src" / "config.py").exists():
                        return cand.resolve()

        for d in depth1_dirs:
            try:
                for sub in d.iterdir():
                    if not sub.is_dir() or sub.name.startswith("."):
                        continue
                    low = sub.name.lower()
                    if "nfl" in low or "bdb" in low:
                        if (sub / "src" / "config.py").exists():
                            return sub.resolve()
                    elif low in {"ml", "work", "projects", "repos", "github", "colab", "colab notebooks", "study"}:
                        cand = sub / "nfl-bdb2027"
                        if (cand / "src" / "config.py").exists():
                            return cand.resolve()
            except OSError:
                continue

    diag = [f"src/config.py를 찾지 못했습니다 (현재 커널 cwd={cwd}, colab={in_colab})."]
    if in_colab:
        for base in (Path("/content"), mydrive):
            if base.exists():
                try:
                    entries = sorted(p.name for p in base.iterdir() if not p.name.startswith("."))[:20]
                except OSError:
                    entries = ["<접근 불가>"]
                diag.append(f"  - {base} 폴더 목록: {entries}")
        diag.extend([
            "",
            "[원인] VS Code에서 Colab 커널(ipykernel)을 선택하면 코드는 로컬 PC가 아니라",
            "       원격 Google Colab VM(/content)에서 실행되므로 로컬 폴더를 직접 읽을 수 없습니다.",
            "[해결 방법]",
            "  1) 로컬 PC 파일로 실행: VS Code 우측 상단 커널을 Colab 대신 로컬 '.venv'로 변경",
            "  2) Colab 커널로 실행: Google Drive(/content/drive/MyDrive/nfl-bdb2027)에 저장소를 두거나,",
            '     다른 Drive 경로에 있다면 이 셀 상단에 os.environ["NFL_PROJECT_ROOT"] = "/content/drive/MyDrive/..." 지정',
        ])
    else:
        diag.append("환경변수 NFL_PROJECT_ROOT를 지정하거나 프로젝트 폴더 안에서 실행하세요.")
    raise FileNotFoundError("\n".join(diag))

ROOT = find_project_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from src import config
from src.features import preprocessing as prep
config.setup()

### [1] 마스터 테이블 조인 (T2 + T3 + T4 → T5)
- **데이터 범위**: `컴바인+경기 통합`
- **작업 단계**: `데이터 통합`
- **목적 및 원리**: 510명 선수를 기준으로 컴바인 피처(T2)와 루키 경기 운동능력(T3)·역할/생산성(T4)을 Left Join한다.
- **확인할 내용**: 정확히 510행, 중복 `nfl_id` 0건.

In [ ]:
# =====================================================================
# [데이터: 컴바인+경기 통합] [단계: 데이터 통합] T5 마스터 테이블 생성
# =====================================================================
master = prep.build_master_table(save=True)
trials = pd.read_parquet(config.PROCESSED_FILES["combine_trials"])
display(master[["nfl_id", "display_name", "combine_position", "sub_position", "draft_year", "g_smax_top5mean", "g_n_plays", "low_exposure", "rookie_snaps_reg"]].head(5))

### [2] 포지션 × 드래프트 티어 층화 80/20 분할 (테스트셋 봉인)
- **데이터 범위**: `컴바인+경기 통합`
- **작업 단계**: `데이터 분할 (누수 차단)`
- **목적 및 원리**: 모든 결측 대치·스케일링·Z-score 비교 전에 테스트셋 102명(`data/splits/test_ids.csv`)을 먼저 완전히 격리한다.
- **확인할 내용**: Train 408명 / Test 102명, 포지션·티어 비율 균일.

In [ ]:
# =====================================================================
# [데이터: 컴바인+경기 통합] [단계: 데이터 분할] 80/20 층화 고정 분할 + 타깃 누수 검증
# =====================================================================
train_df, test_df = prep.create_or_load_test_split(master, overwrite=False)
prep.check_relative_labels_ignore_holdout(train_df, test_df)
print("[확인된 결과] 상대 기준 타깃 교란 테스트 통과: 테스트 정답을 바꿔도 Train 라벨 불변")
display(pd.crosstab(train_df["combine_position"], train_df["draft_tier"], margins=True))

### [3] 결측치 처리 방식 비교 (그룹 중앙값 vs KNN vs MICE)
- **데이터 범위**: `컴바인+경기 통합` (Train 408명 전용)
- **작업 단계**: `결측 방식 비교`
- **목적 및 원리**: 포지션 그룹 중앙값(기준), 포지션 가중 KNN Imputer, 포지션 조건부 MICE(`IterativeImputer`)를 동일한 5×5 CV + Ridge로 비교한다. 세 설정은 **같은 25개 fold**를 쓰므로 fold별 점수 차이로 짝지어 비교한다.
- **확인할 내용**: `delta_mean`(기준 대비 포지션 내 $R^2$ 차이), `wins`(25개 중 이긴 fold 수), `verdict`(우위/차이 없음/열위).

In [ ]:
# =====================================================================
# [데이터: 컴바인+경기 통합] [단계: 결측 방식 비교] Group Median vs KNN vs MICE
# =====================================================================
imp_tbl = prep.compare_imputers_cv(train_df, trials, target_col="g_smax_top5mean")
display(imp_tbl)

### [4] 스케일링 · 분포 변환 방식 비교 (`Robust`(기준) vs `Standard` vs `Yeo-Johnson`)
- **데이터 범위**: `컴바인+경기 통합` (Train 408명 전용)
- **작업 단계**: `스케일링 비교`
- **목적 및 원리**: 이상치에 민감한 `StandardScaler`, 중앙값·IQR 기반 `RobustScaler`, 치우침을 펴주는 `Yeo-Johnson` 변환을 같은 fold로 짝지어 비교한다.
- **확인할 내용**: `verdict` 열. "차이 없음"이면 가장 단순한 방식을 기본값으로 둔다.

In [ ]:
# =====================================================================
# [데이터: 컴바인+경기 통합] [단계: 스케일링 비교] Standard vs Robust vs Yeo-Johnson
# =====================================================================
sc_tbl = prep.compare_scalers_cv(train_df, trials, target_col="g_smax_top5mean")
display(sc_tbl)

### [5] 피처 세트 점진적 효과 비교 (절대 타깃 & 시즌 코호트 상대 타깃)
- **데이터 범위**: `컴바인+경기 통합` (Train 408명 전용)
- **작업 단계**: `피처 추가 효과 검증`
- **목적 및 원리**: `FS-Trad`(기준, 전통 12개) → `FS-Phys`(+체격비율·물리·나이·결측플래그) → `FS-Biomech`(+3축 트래킹·40야드 F-V·스킬 드릴 구제·드릴별 Z-score) → `FS-Full`(+NGS)를 **절대 타깃과 시즌 코호트 상대 타깃** 양쪽에서 짝지어 비교한다. 코호트 타깃은 fold마다 학습 fold로만 기준을 잡는다.
- **확인할 내용**: `FS-Trad` 대비 `delta_mean`과 `verdict`.

In [ ]:
# =====================================================================
# [데이터: 컴바인+경기 통합] [단계: 피처 효과 검증] FS-Trad -> FS-Full
# =====================================================================
fs_tbl = prep.compare_feature_sets_cv(train_df, trials)
display(fs_tbl)

### [6] 차원 · 피처 생존 점검 (PCA 설명분산 & Lasso / ElasticNet 계수)
- **데이터 범위**: `컴바인+경기 통합` (Train 408명 전용)
- **작업 단계**: `차원·피처 점검` + `인코딩 검증`
- **목적 및 원리**: 이미 학습한 PCA로 유효 차원 수(80%·90% 분산)를 확인하고, Lasso/ElasticNet에서 어떤 피처가 살아남는지 확인한다. 코호트 타깃은 Train 408명으로만 기준을 잡는다.
- **확인할 내용**: 상위 20개 생존 피처 및 계수 부호의 물리적 타당성.

In [ ]:
# =====================================================================
# [데이터: 컴바인+경기 통합] [단계: 차원·피처 점검] PCA + Lasso/ElasticNet
# =====================================================================
pca_tbl, coef_tbl = prep.inspect_pca_and_lasso(train_df, trials, target_col="g_smax_top5mean_cohort_z")
display(pca_tbl.head(15))
display(coef_tbl.head(20))